In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

In [8]:
df = pd.read_csv('ParisHousing.csv')

df.shape
# df.describe()

(10000, 17)

In [19]:
# коробочное решение

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import numpy as np

X = df.drop(["price", "cityCode"], axis=1)
y = df["price"]

# делим на train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# нормализуем (scaler учится на train, применяется к обоим)
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm  = scaler.transform(X_test)

# обучаем
model = LinearRegression()
model.fit(X_train_norm, y_train)

# оцениваем
predictions = model.predict(X_test_norm)
r2 = r2_score(y_test, predictions)
mean_error = np.abs(predictions - y_test).mean()
mape = mean_absolute_percentage_error(y_test, predictions)

print("веса:", [round(c, 3) for c in model.coef_], " bias:", round(model.intercept_, 3))
print("R2 на тесте:", round(r2, 3))
print("средняя ошибка:", f"${round(mean_error):,}")
print("средняя абсолютная процентная ошибка:", f"{round(mape * 100, 2)}%")

веса: [np.float64(2855617.887), np.float64(7.744), np.float64(1496.495), np.float64(1496.598), np.float64(1575.849), np.float64(144.897), np.float64(8.295), np.float64(-30.098), np.float64(64.251), np.float64(51.343), np.float64(-23.509), np.float64(-22.119), np.float64(26.592), np.float64(20.767), np.float64(-19.818)]  bias: 4964091.749
R2 на тесте: 1.0
средняя ошибка: $1,509
средняя абсолютная процентная ошибка: 0.09%


In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.pipeline import make_pipeline

import numpy as np

X = df.drop(["price", "cityCode"], axis=1)
y = df["price"]

# scaler + модель в одном пайплайне (чтобы нормализация была честной внутри каждого фолда)
model = make_pipeline(StandardScaler(), LinearSVR(max_iter=10000, C=850.0))

# cv=5 → делим данные на 5 частей, 5 раз учим и проверяем
scores = cross_val_score(model, X, y, cv=5, scoring="r2")
predictions = cross_val_predict(model, X, y, cv=5)

# оцениваем
mape = mean_absolute_percentage_error(y, predictions)

print("средняя абсолютная процентная ошибка:", f"{round(mape * 100, 2)}%")
print("R2 на кросс-валидации:", round(scores.mean(), 3))


средняя абсолютная процентная ошибка: 2.58%
R2 на кросс-валидации: 0.999


In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(["price", "cityCode"], axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test) # запомнить

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(15,)),
    tf.keras.layers.Dense(90, activation='relu'),
    tf.keras.layers.Dense(45, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam',
              loss='mse',
              metrics=['mae', 'r2_score'])

# 5. Обучаем модель ТОЛЬКО на тренировочных данных
history = model.fit(X_train_scaled, y_train_scaled, 
                    epochs=100, 
                    batch_size=32, 
                    validation_data=(X_test_scaled, y_test_scaled))

# 6. Финальная оценка модели на тестовых данных
loss, mae, r2_score = model.evaluate(X_test_scaled, y_test_scaled)
print(f"\nСредняя абсолютная ошибка (MAE) на тесте (в масштабе): {mae:.4f}")
print(f"Коэффициент детерминации (R²) на тесте (в масштабе): {r2_score:.4f}")


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 794us/step - loss: 0.1266 - mae: 0.2180 - r2_score: 0.8734 - val_loss: 0.0120 - val_mae: 0.0866 - val_r2_score: 0.9888
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 506us/step - loss: 0.0093 - mae: 0.0765 - r2_score: 0.9907 - val_loss: 0.0080 - val_mae: 0.0707 - val_r2_score: 0.9926
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 535us/step - loss: 0.0064 - mae: 0.0638 - r2_score: 0.9936 - val_loss: 0.0061 - val_mae: 0.0622 - val_r2_score: 0.9943
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 521us/step - loss: 0.0046 - mae: 0.0541 - r2_score: 0.9954 - val_loss: 0.0052 - val_mae: 0.0575 - val_r2_score: 0.9952
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 531us/step - loss: 0.0035 - mae: 0.0470 - r2_score: 0.9965 - val_loss: 0.0038 - val_mae: 0.0494 - val_r2_score: 0.9964
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 547us/step - loss: 0.0027 - mae: 0.0418 - r2_score: 0.9973 - val_loss: 0.0032 - val_mae: 0.0452 - val_r2_score: 0.9970
Epoch 7/100
250/250 ━━

array([[7.9815705e+06],
       [3.7618645e+06],
       [8.5556500e+06],
       [6.4397390e+06],
       [6.4937285e+06]], dtype=float32)